# Dashboard Clase 16 — Evaluación Rigurosa de Clasificadores

**Diplomado en Data Science Aplicada con Python para la Toma de Decisiones**  
Arca Continental Ecuador | UDLA · *15 de Abril, 2026*

Dashboard interactivo con Dash + Plotly que contiene **todo** lo visto hoy:

1. **Overview** — dataset y desbalance
2. **EDA interactivo** — explora las variables con plots 2D y 3D
3. **Train/Test + curvas ROC y PR**
4. **🎯 Umbral interactivo** — *mueve el slider* y mira cómo cambian la matriz de confusión y las métricas en tiempo real
5. **Interpretabilidad** — coeficientes, odds ratios y correlaciones

---

## Instalación (una sola vez)

```
!pip install -q dash jupyter-dash plotly scikit-learn pandas numpy
```

In [ ]:
# Descomenten la siguiente linea la primera vez que corran en Colab
# !pip install -q dash jupyter-dash

## 1. Carga de datos y entrenamiento del modelo

In [ ]:
import pandas as pd
import numpy as np
import base64
import urllib.request
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score,
                             precision_score, recall_score, accuracy_score, f1_score)

# Paleta Arca
ARCA_RED   = "#C82B40"
ARCA_DARK  = "#6B1525"
UDLA_RED   = "#A31545"
ARCA_GRAY  = "#F5F5F5"
BLUE       = "#2563EB"
GREEN      = "#16A34A"
ORANGE     = "#EA580C"

# ----- Datos -----
URL = "https://raw.githubusercontent.com/cmosquerat/arca-diplomado/main/clase-16/stroke.csv"
df = pd.read_csv(URL)
df_raw = df.copy()
df = df.drop(columns=["id"])
df["bmi"] = df["bmi"].fillna(df["bmi"].median())

df_enc = pd.get_dummies(df, drop_first=True, dtype=int)
X = df_enc.drop(columns=["stroke"])
y = df_enc["stroke"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

modelo = LogisticRegression(max_iter=1000, class_weight="balanced")
modelo.fit(X_train_s, y_train)
y_proba = modelo.predict_proba(X_test_s)[:, 1]

auc_global = roc_auc_score(y_test, y_proba)
ap_global  = average_precision_score(y_test, y_proba)

coefs = pd.DataFrame({"feature": X.columns, "coef": modelo.coef_[0]})
coefs["odds_ratio"] = np.exp(coefs["coef"])
coefs["abs"] = coefs["coef"].abs()
coefs = coefs.sort_values("coef")

corr_target = df_enc.corr(numeric_only=True)["stroke"].drop("stroke").sort_values()
corr_matrix = df_enc.corr(numeric_only=True)

print(f"Dataset: {len(df_raw)} filas — {y.mean()*100:.2f}% positivos")
print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"AUC = {auc_global:.3f} | AP = {ap_global:.3f}")

## 2. Logos de Arca Continental y UDLA (embebidos)

In [ ]:
def image_to_base64(url_or_path):
    """Carga imagen desde URL (raw github) o archivo local, devuelve data URI."""
    if url_or_path.startswith("http"):
        data = urllib.request.urlopen(url_or_path).read()
    else:
        with open(url_or_path, "rb") as f:
            data = f.read()
    ext = url_or_path.split(".")[-1].lower()
    return f"data:image/{ext};base64,{base64.b64encode(data).decode()}"

LOGO_ARCA = image_to_base64(
    "https://raw.githubusercontent.com/cmosquerat/arca-diplomado/main/ArcaContinental.png")
LOGO_UDLA = image_to_base64(
    "https://raw.githubusercontent.com/cmosquerat/arca-diplomado/main/logo_udla.png")

print("Logos cargados.")

## 3. Dashboard

La siguiente celda lanza el dashboard **dentro del notebook**. Si usas Colab, se renderiza inline. Si estás local, se abre en http://127.0.0.1:8050 por defecto.

In [ ]:
from dash import Dash, dcc, html, Input, Output, callback

# ============================================================================
#  ESTILOS
# ============================================================================
CARD = {
    "backgroundColor": "white", "borderRadius": "12px",
    "padding": "20px", "margin": "10px",
    "boxShadow": "0 2px 10px rgba(107,21,37,0.08)",
    "border": f"1px solid {ARCA_GRAY}",
}
METRIC = {
    "backgroundColor": "white", "borderRadius": "12px",
    "padding": "20px", "margin": "8px", "textAlign": "center",
    "border": f"2px solid {ARCA_RED}",
    "boxShadow": "0 2px 8px rgba(200,43,64,0.15)",
    "flex": "1",
}

def metric_card(label, value, color=ARCA_DARK):
    return html.Div([
        html.Div(label, style={"color": "#6B7280", "fontSize": "13px",
                               "fontWeight": "600", "textTransform": "uppercase",
                               "letterSpacing": "0.5px"}),
        html.Div(value, style={"color": color, "fontSize": "32px",
                               "fontWeight": "700", "marginTop": "4px"}),
    ], style=METRIC)

# ============================================================================
#  GRAFICOS ESTATICOS (no dependen del umbral)
# ============================================================================
def fig_target():
    counts = df["stroke"].value_counts().rename({0: "Sano", 1: "ACV"})
    fig = go.Figure(go.Bar(
        x=counts.index, y=counts.values,
        marker_color=[BLUE, ARCA_RED], text=counts.values, textposition="outside"))
    fig.update_layout(title="Distribución del target (desbalance 95/5)",
                      template="plotly_white", height=350,
                      margin=dict(l=40, r=20, t=50, b=40))
    return fig

def fig_age_stroke():
    dd = df.copy()
    dd["stroke_lbl"] = dd["stroke"].map({0: "Sano", 1: "ACV"})
    fig = px.violin(dd, x="stroke_lbl", y="age", color="stroke_lbl",
                    box=True, points="outliers",
                    color_discrete_map={"Sano": BLUE, "ACV": ARCA_RED},
                    title="Edad por estado de ACV")
    fig.update_layout(template="plotly_white", height=400, showlegend=False,
                      margin=dict(l=40, r=20, t=50, b=40))
    return fig

def fig_3d():
    dd = df.sample(n=min(2000, len(df)), random_state=1).copy()
    dd["stroke_lbl"] = dd["stroke"].map({0: "Sano", 1: "ACV"})
    fig = px.scatter_3d(dd, x="age", y="avg_glucose_level", z="bmi",
                        color="stroke_lbl", opacity=0.55,
                        color_discrete_map={"Sano": BLUE, "ACV": ARCA_RED},
                        title="Edad × Glucosa × BMI — rota con el mouse")
    fig.update_layout(template="plotly_white", height=520,
                      margin=dict(l=0, r=0, t=40, b=0))
    return fig

def fig_heatmap_eda():
    fig = px.density_heatmap(df, x="age", y="avg_glucose_level",
                             nbinsx=25, nbinsy=25,
                             color_continuous_scale="Reds",
                             title="Densidad: edad vs glucosa")
    fig.update_layout(template="plotly_white", height=400,
                      margin=dict(l=40, r=20, t=50, b=40))
    return fig

def fig_roc():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", fill="tozeroy",
                             line=dict(color=ARCA_RED, width=3),
                             fillcolor="rgba(200,43,64,0.15)",
                             name=f"Modelo (AUC={auc_global:.3f})"))
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                             line=dict(color="gray", width=1.5, dash="dash"),
                             name="Azar"))
    fig.update_layout(title=f"Curva ROC (AUC = {auc_global:.3f})",
                      xaxis_title="FPR", yaxis_title="TPR (Recall)",
                      template="plotly_white", height=430,
                      margin=dict(l=40, r=20, t=50, b=40))
    return fig

def fig_pr():
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=rec, y=prec, mode="lines", fill="tozeroy",
                             line=dict(color=ARCA_RED, width=3),
                             fillcolor="rgba(200,43,64,0.15)",
                             name=f"Modelo (AP={ap_global:.3f})"))
    fig.add_hline(y=y_test.mean(), line_dash="dash", line_color="gray",
                  annotation_text=f"baseline = {y_test.mean():.3f}")
    fig.update_layout(title=f"Curva Precision-Recall (AP = {ap_global:.3f})",
                      xaxis_title="Recall", yaxis_title="Precision",
                      template="plotly_white", height=430,
                      margin=dict(l=40, r=20, t=50, b=40))
    return fig

def fig_coefs():
    fig = px.bar(coefs, x="coef", y="feature", orientation="h",
                 color="coef", color_continuous_scale="RdBu_r",
                 color_continuous_midpoint=0,
                 title="Coeficientes estandarizados (rojo = empuja hacia ACV)")
    fig.add_vline(x=0, line_dash="dash", line_color="gray")
    fig.update_layout(template="plotly_white", height=550,
                      margin=dict(l=40, r=20, t=50, b=40))
    return fig

def fig_odds():
    top = coefs.reindex(coefs["abs"].sort_values().index).tail(10).copy()
    top["efecto"] = np.where(top["odds_ratio"] > 1, "Aumenta riesgo", "Reduce riesgo")
    fig = px.bar(top, x="odds_ratio", y="feature", orientation="h",
                 color="efecto",
                 color_discrete_map={"Aumenta riesgo": ARCA_RED,
                                     "Reduce riesgo": BLUE},
                 title="Top 10 variables — Odds Ratios")
    fig.add_vline(x=1, line_dash="dash", line_color="gray",
                  annotation_text="OR = 1 (sin efecto)")
    fig.update_layout(template="plotly_white", height=450,
                      margin=dict(l=40, r=20, t=50, b=40))
    return fig

def fig_corr_target():
    fig = px.bar(x=corr_target.values, y=corr_target.index, orientation="h",
                 color=corr_target.values, color_continuous_scale="RdBu_r",
                 color_continuous_midpoint=0,
                 title="Correlación de cada variable con stroke")
    fig.update_layout(template="plotly_white", height=550,
                      margin=dict(l=40, r=20, t=50, b=40),
                      xaxis_title="Correlación", yaxis_title="")
    return fig

def fig_corr_matrix():
    fig = px.imshow(corr_matrix, color_continuous_scale="RdBu_r",
                    color_continuous_midpoint=0, aspect="auto",
                    title="Matriz de correlación entre features")
    fig.update_layout(template="plotly_white", height=650,
                      margin=dict(l=40, r=20, t=50, b=40))
    fig.update_xaxes(tickangle=45)
    return fig

# ============================================================================
#  LAYOUT
# ============================================================================

HEADER = html.Div([
    html.Div([
        html.Img(src=LOGO_ARCA, style={"height": "55px"}),
        html.Div([
            html.H1("Evaluación Rigurosa de Clasificadores",
                    style={"margin": "0", "color": "white",
                           "fontSize": "26px", "fontWeight": "700"}),
            html.Div("Clase 16 · Predicción de ACV · Diplomado Arca Continental × UDLA",
                     style={"color": "rgba(255,255,255,0.85)",
                            "fontSize": "14px", "marginTop": "3px"}),
        ], style={"flex": "1", "marginLeft": "20px"}),
        html.Img(src=LOGO_UDLA, style={"height": "55px"}),
    ], style={"display": "flex", "alignItems": "center",
              "padding": "18px 30px"}),
], style={"background": f"linear-gradient(135deg, {ARCA_DARK} 0%, {ARCA_RED} 100%)",
          "boxShadow": "0 4px 20px rgba(0,0,0,0.15)"})

TAB_STYLE = {"padding": "12px 20px", "fontWeight": "600",
             "backgroundColor": "white", "borderRadius": "0"}
TAB_SELECTED = {"padding": "12px 20px", "fontWeight": "700",
                "backgroundColor": ARCA_RED, "color": "white",
                "borderTop": f"3px solid {ARCA_DARK}"}

def tab_overview():
    return html.Div([
        html.Div([
            metric_card("Total pacientes", f"{len(df_raw):,}"),
            metric_card("Casos de ACV", f"{int(y.sum())}", ARCA_RED),
            metric_card("% positivos", f"{y.mean()*100:.2f}%", ARCA_RED),
            metric_card("Features usadas", f"{X.shape[1]}"),
            metric_card("Train / Test", f"{len(X_train)} / {len(X_test)}"),
        ], style={"display": "flex", "flexWrap": "wrap"}),
        html.Div([
            html.Div(dcc.Graph(figure=fig_target()), style=CARD),
            html.Div([
                html.H3("Contexto clínico", style={"color": ARCA_DARK, "marginTop": 0}),
                html.P([
                    "Un hospital quiere identificar pacientes con alto riesgo de ACV ",
                    "(accidente cerebrovascular) para prevenir antes de que ocurra. "
                ]),
                html.Ul([
                    html.Li("Desbalance severo: solo 4.87% de positivos."),
                    html.Li("Features: edad, hipertensión, enfermedad cardíaca, glucosa, BMI, tabaquismo, tipo de trabajo."),
                    html.Li([html.B("Asimetría de costos:"), " un falso negativo puede costar una vida; un falso positivo cuesta un chequeo."]),
                ]),
                html.Div([
                    html.Span("AUC global: ", style={"fontWeight": 600}),
                    html.Span(f"{auc_global:.3f}", style={"color": ARCA_RED, "fontWeight": 700}),
                    html.Span("  ·  AP global: ", style={"fontWeight": 600, "marginLeft": "12px"}),
                    html.Span(f"{ap_global:.3f}", style={"color": ARCA_RED, "fontWeight": 700}),
                ], style={"marginTop": "14px", "fontSize": "16px"}),
            ], style={**CARD, "flex": "1"}),
        ], style={"display": "flex", "flexWrap": "wrap"}),
    ])

def tab_eda():
    return html.Div([
        html.Div([
            html.Div(dcc.Graph(figure=fig_age_stroke()), style={**CARD, "flex": "1"}),
            html.Div(dcc.Graph(figure=fig_heatmap_eda()), style={**CARD, "flex": "1"}),
        ], style={"display": "flex", "flexWrap": "wrap"}),
        html.Div(dcc.Graph(figure=fig_3d()), style=CARD),
    ])

def tab_evaluation():
    return html.Div([
        html.Div([
            html.Div(dcc.Graph(figure=fig_roc()), style={**CARD, "flex": "1"}),
            html.Div(dcc.Graph(figure=fig_pr()), style={**CARD, "flex": "1"}),
        ], style={"display": "flex", "flexWrap": "wrap"}),
        html.Div([
            html.H3("Cómo leer estas curvas", style={"color": ARCA_DARK, "marginTop": 0}),
            html.Ul([
                html.Li([html.B("ROC: "), "TPR vs FPR para cada umbral. AUC = área bajo la curva. Más pegada arriba-izquierda = mejor."]),
                html.Li([html.B("PR: "), "Precision vs Recall. La línea punteada es el modelo aleatorio (prevalencia del target)."]),
                html.Li([html.B("En desbalance, "), html.Span("la PR es más honesta", style={"color": ARCA_RED, "fontWeight": 700}), " que la ROC."]),
            ]),
        ], style=CARD),
    ])

def tab_threshold():
    return html.Div([
        html.Div([
            html.H3("🎯 Umbral interactivo — mueve el slider",
                    style={"color": ARCA_DARK, "marginTop": 0}),
            html.P("Cada posición del slider redefine qué probabilidades se clasifican como ACV. "
                   "La matriz de confusión y las métricas se recalculan en tiempo real."),
            dcc.Slider(id="threshold-slider", min=0.05, max=0.95, step=0.01, value=0.5,
                       marks={i/10: f"{i/10:.1f}" for i in range(1, 10)},
                       tooltip={"placement": "top", "always_visible": True}),
        ], style=CARD),
        html.Div(id="threshold-metrics", style={"display": "flex", "flexWrap": "wrap"}),
        html.Div([
            html.Div(dcc.Graph(id="confusion-matrix"), style={**CARD, "flex": "1"}),
            html.Div(dcc.Graph(id="threshold-distribution"), style={**CARD, "flex": "1"}),
        ], style={"display": "flex", "flexWrap": "wrap"}),
        html.Div(dcc.Graph(id="threshold-on-curves"), style=CARD),
    ])

def tab_interpret():
    return html.Div([
        html.Div([
            html.Div(dcc.Graph(figure=fig_coefs()), style={**CARD, "flex": "1"}),
            html.Div(dcc.Graph(figure=fig_corr_target()), style={**CARD, "flex": "1"}),
        ], style={"display": "flex", "flexWrap": "wrap"}),
        html.Div([
            html.Div(dcc.Graph(figure=fig_odds()), style={**CARD, "flex": "1"}),
            html.Div(dcc.Graph(figure=fig_corr_matrix()), style={**CARD, "flex": "1"}),
        ], style={"display": "flex", "flexWrap": "wrap"}),
        html.Div([
            html.H3("Lectura clínica", style={"color": ARCA_DARK, "marginTop": 0}),
            html.Ul([
                html.Li([html.B("age domina "), "con coef ≈ 1.9 y OR ≈ 6.6. Por cada desviación estándar de edad, las odds se multiplican por ~6.6."]),
                html.Li([html.B("hypertension, avg_glucose_level, heart_disease "), "son factores modificables — donde la intervención tiene sentido."]),
                html.Li([html.B("ever_married_Yes "), "parecía correlacionar (0.11) pero su coeficiente cae a casi 0 — era confounder de edad."]),
            ]),
        ], style=CARD),
    ])

# ============================================================================
#  APP
# ============================================================================
app = Dash(__name__, suppress_callback_exceptions=True)

app.layout = html.Div([
    HEADER,
    dcc.Tabs(id="tabs", value="overview", children=[
        dcc.Tab(label="1. Overview", value="overview",
                style=TAB_STYLE, selected_style=TAB_SELECTED),
        dcc.Tab(label="2. EDA", value="eda",
                style=TAB_STYLE, selected_style=TAB_SELECTED),
        dcc.Tab(label="3. ROC & PR", value="eval",
                style=TAB_STYLE, selected_style=TAB_SELECTED),
        dcc.Tab(label="4. Umbral interactivo 🎯", value="thr",
                style=TAB_STYLE, selected_style=TAB_SELECTED),
        dcc.Tab(label="5. Interpretabilidad", value="interp",
                style=TAB_STYLE, selected_style=TAB_SELECTED),
    ]),
    html.Div(id="tab-content", style={"padding": "10px", "backgroundColor": "#FAFAFA"}),
], style={"fontFamily": "'Segoe UI', 'Helvetica Neue', sans-serif",
          "backgroundColor": "#FAFAFA", "minHeight": "100vh"})

@callback(Output("tab-content", "children"), Input("tabs", "value"))
def render_tab(t):
    return {"overview": tab_overview(), "eda": tab_eda(),
            "eval": tab_evaluation(), "thr": tab_threshold(),
            "interp": tab_interpret()}.get(t, tab_overview())

# ---- Callback: umbral interactivo ----
@callback(
    Output("threshold-metrics", "children"),
    Output("confusion-matrix", "figure"),
    Output("threshold-distribution", "figure"),
    Output("threshold-on-curves", "figure"),
    Input("threshold-slider", "value"))
def update_threshold(thr):
    y_pred = (y_proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    p = tp / (tp + fp) if (tp + fp) else 0
    r = tp / (tp + fn) if (tp + fn) else 0
    acc = (tp + tn) / (tp + tn + fp + fn)
    f1 = 2 * p * r / (p + r) if (p + r) else 0

    metrics = [
        metric_card("Umbral", f"{thr:.2f}", ARCA_DARK),
        metric_card("Precision", f"{p:.2%}", ARCA_RED),
        metric_card("Recall", f"{r:.2%}", ARCA_RED),
        metric_card("F1", f"{f1:.2%}", ARCA_DARK),
        metric_card("Accuracy", f"{acc:.2%}", ARCA_DARK),
    ]

    cm = np.array([[tn, fp], [fn, tp]])
    labels = [["VN", "FP"], ["FN", "VP"]]
    annotations = [[f"{labels[i][j]}<br><b>{cm[i][j]}</b>" for j in range(2)] for i in range(2)]
    cm_fig = go.Figure(go.Heatmap(
        z=cm, x=["Predicho: Sano", "Predicho: ACV"], y=["Real: Sano", "Real: ACV"],
        colorscale="Reds", text=annotations, texttemplate="%{text}",
        textfont=dict(size=18), showscale=False))
    cm_fig.update_layout(title=f"Matriz de confusión — umbral {thr:.2f}",
                          template="plotly_white", height=430,
                          margin=dict(l=40, r=20, t=50, b=40))
    cm_fig.update_yaxes(autorange="reversed")

    dist = go.Figure()
    dist.add_trace(go.Histogram(x=y_proba[y_test == 0], name="Sano",
                                marker_color=BLUE, opacity=0.7, nbinsx=40))
    dist.add_trace(go.Histogram(x=y_proba[y_test == 1], name="ACV",
                                marker_color=ARCA_RED, opacity=0.7, nbinsx=40))
    dist.add_vline(x=thr, line_color=ARCA_DARK, line_width=3,
                   annotation_text=f"umbral {thr:.2f}")
    dist.update_layout(title="Distribución de probabilidades y corte del umbral",
                       barmode="overlay", template="plotly_white", height=430,
                       margin=dict(l=40, r=20, t=50, b=40),
                       xaxis_title="Probabilidad predicha")

    fpr_arr, tpr_arr, _ = roc_curve(y_test, y_proba)
    prec_arr, rec_arr, _ = precision_recall_curve(y_test, y_proba)
    fpr_pt = fp / (fp + tn) if (fp + tn) else 0
    tpr_pt = r

    curves = make_subplots(rows=1, cols=2, subplot_titles=("ROC", "Precision-Recall"))
    curves.add_trace(go.Scatter(x=fpr_arr, y=tpr_arr, mode="lines",
                                line=dict(color=ARCA_RED, width=3),
                                fill="tozeroy", fillcolor="rgba(200,43,64,0.12)",
                                showlegend=False), row=1, col=1)
    curves.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                                line=dict(color="gray", dash="dash"),
                                showlegend=False), row=1, col=1)
    curves.add_trace(go.Scatter(x=[fpr_pt], y=[tpr_pt], mode="markers",
                                marker=dict(color=ARCA_DARK, size=18,
                                            line=dict(color="white", width=2)),
                                name=f"Umbral {thr:.2f}",
                                showlegend=False), row=1, col=1)
    curves.add_trace(go.Scatter(x=rec_arr, y=prec_arr, mode="lines",
                                line=dict(color=ARCA_RED, width=3),
                                fill="tozeroy", fillcolor="rgba(200,43,64,0.12)",
                                showlegend=False), row=1, col=2)
    curves.add_trace(go.Scatter(x=[r], y=[p], mode="markers",
                                marker=dict(color=ARCA_DARK, size=18,
                                            line=dict(color="white", width=2)),
                                showlegend=False), row=1, col=2)
    curves.update_xaxes(title_text="FPR", row=1, col=1, range=[0, 1])
    curves.update_yaxes(title_text="TPR", row=1, col=1, range=[0, 1])
    curves.update_xaxes(title_text="Recall", row=1, col=2, range=[0, 1])
    curves.update_yaxes(title_text="Precision", row=1, col=2, range=[0, 1])
    curves.update_layout(title=f"Dónde cae el umbral {thr:.2f} sobre las curvas",
                         template="plotly_white", height=430,
                         margin=dict(l=40, r=20, t=70, b=40))

    return metrics, cm_fig, dist, curves

# Lanzar
app.run(debug=False, jupyter_mode="inline", jupyter_height=1100)

---

## Notas

- Si el dashboard no se renderiza inline en Colab, cambien `jupyter_mode="inline"` por `jupyter_mode="external"`. Se abrirá en una pestaña nueva.
- Para detener la app, reinicien el kernel del notebook.
- El logo de Arca Continental y UDLA se cargan desde el mismo repo del diplomado.
- **Todo** lo visto hoy está aquí: overview del problema, EDA, evaluación ROC/PR, umbral interactivo y análisis de interpretabilidad.